# Time-series forecasting with SAP-RPT-TS
## Multiple time-series
This notebook belogs to the <a href='https://community.sap.com/t5/technology-blog-posts-by-sap/hands-on-tutorial-improve-your-time-series-forecasts-with-sap-s-tabular-ai/ba-p/14514276'>Hands-on Tutorial: Improve your time-series forecasts with SAP's Tabular AI model SAP-RPT-TS</a>

### SAP AI Core details

Service key for SAP AI Core and url of deployed model

In [ ]:
# Service Key for SAP AI Core, downloaded from the BTP Cockpit
aicore_servicekey_file_path = 'aicore-servicekey.json'

# Deployment url of the deployed model on SAP AI Core
# SAP AI Launchpad -> ML Operations -> Deployments -> Click the model and you see the URL
DEPLOYMENT_URL_SAPRPTTS = "YOURURLHERE"

In [2]:
import json
from pathlib import Path
aicore_servicekey = Path(aicore_servicekey_file_path)
with aicore_servicekey.open() as f:
    sk = json.load(f)

AICORE_AUTH_URL      = sk["url"]
AICORE_CLIENT_ID     = sk["clientid"]
AICORE_CLIENT_SECRET = sk["clientsecret"]

Obtain Bearer token to access SAP AI Core API

In [3]:
import os
import requests
response = requests.post(
    f"{AICORE_AUTH_URL}/oauth/token",
    headers={"Content-Type": "application/x-www-form-urlencoded"},
    data={
        "grant_type": "client_credentials",
        "client_id": AICORE_CLIENT_ID,
        "client_secret": AICORE_CLIENT_SECRET,
    },
)
response.raise_for_status()
AUTH_TOKEN = response.json()["access_token"]

### Access and prepare data
Connect with SAP HANA Cloud (or SAP Datasphere) to access the historic data. 
<BR>The table can be added to your system, see <a hrf='https://community.sap.com/t5/technology-blog-posts-by-sap/hands-on-tutorial-machine-learning-with-sap-hana-cloud-and-bas/ba-p/14404096'>Hands-on Tutorial: Machine Learning with SAP HANA Cloud and BAS</a>

In [4]:
import json
import hana_ml.dataframe as dataframe
with open('./hanacloud_credentials.json', 'r') as creds:
  credentials = json.load(creds)
SAP_HANA_CLOUD_ADDRESS  = credentials["SAP_HANA_CLOUD"]["HANA_ADDRESS"]
SAP_HANA_CLOUD_PORT     = credentials["SAP_HANA_CLOUD"]["HANA_PORT"]
SAP_HANA_CLOUD_USER     = credentials["SAP_HANA_CLOUD"]["HANA_USER"]
SAP_HANA_CLOUD_PASSWORD = credentials["SAP_HANA_CLOUD"]["HANA_PASSWORD"]
conn = dataframe.ConnectionContext(address  = SAP_HANA_CLOUD_ADDRESS,
                                   port     = SAP_HANA_CLOUD_PORT,
                                   user     = SAP_HANA_CLOUD_USER,
                                   password = SAP_HANA_CLOUD_PASSWORD)
conn.connection.isconnected()

True

Create hana_ml DataFrame that points to the SAP HANA Cloud table

In [5]:
data_hdf = conn.table('HOTELNIGHTS')
data_hdf.head(5).collect()

,MONTH,REGION,COUNTRYOFRESIDENCE,ARRIVALS,HOTELNIGHTS
0,2020-01-01,Zurich Region,Austria,3974,6635
1,2020-01-01,Zurich Region,Norway,534,994
2,2020-01-01,Zurich Region,Portugal,1233,2695
3,2020-01-01,Zurich Region,Other European countries,1184,2393
4,2020-01-01,Zurich Region,Poland,1516,3726


Remove record with incorrect country assignment

In [6]:
data_hdf = data_hdf.filter("COUNTRYOFRESIDENCE NOT IN ('Australia, New Zealand, Oceania')")
data_hdf.select('COUNTRYOFRESIDENCE').distinct().count()

72

Brief data exploration

In [7]:
data_hdf.select('MONTH').describe().collect()

,column,count,unique,nulls,mean,std,min,max,min_value,max_value,median,25_percent_cont,25_percent_disc,50_percent_cont,50_percent_disc,75_percent_cont,75_percent_disc
0,MONTH,67756,75,0,None,None,2020-01-01,2026-03-01,2020-01-01,2026-03-01,None,None,None,None,None,None,None


Aggregate and sort by COUNTRYOFRESIDENCE and MONTH

In [8]:
agg_hdf = data_hdf.agg([('sum', 'HOTELNIGHTS', 'HOTELNIGHTS')], group_by=['COUNTRYOFRESIDENCE', 'MONTH'])
agg_hdf = agg_hdf.sort(['COUNTRYOFRESIDENCE', 'MONTH'])
agg_hdf.head(5).collect()

,COUNTRYOFRESIDENCE,MONTH,HOTELNIGHTS
0,Argentina,2020-01-01,3552
1,Argentina,2020-02-01,3160
2,Argentina,2020-03-01,1026
3,Argentina,2020-04-01,130
4,Argentina,2020-05-01,55


In [9]:
agg_hdf.count()

5399

Plot select time-series

In [10]:
import plotly.express as px
country_filter = 'Switzerland'
data_df = agg_hdf.filter(f"COUNTRYOFRESIDENCE = '{country_filter}'").collect()
fig = px.line(data_df, x="MONTH", y="HOTELNIGHTS", title=f'Hotel nights {country_filter}')
fig.show()

Drop data before 2022 to eliminate Covid time

In [11]:
agg_hdf = agg_hdf.filter("YEAR(MONTH) >= 2022")
agg_hdf.count()

3672

In [12]:
agg_hdf.select('MONTH').describe().collect()

,column,count,unique,nulls,mean,std,min,max,min_value,max_value,median,25_percent_cont,25_percent_disc,50_percent_cont,50_percent_disc,75_percent_cont,75_percent_disc
0,MONTH,3672,51,0,None,None,2022-01-01,2026-03-01,2022-01-01,2026-03-01,None,None,None,None,None,None,None


Keep the last 12 months as holdout. Predict the known past to assess forecast accuracy

In [13]:
train_hdf = agg_hdf.filter("MONTH < '2025-04-01' ")
holdout_hdf = agg_hdf.filter("MONTH >= '2025-04-01' ")
train_hdf.count()

2808

In [14]:
holdout_hdf.count()

864

### Get forecast from SAP-RPT-TS on SAP AI Core

Model data as needed by SAP-RPT-TS, row-wise for each time-series

In [15]:
train_piv_hdf = train_hdf.pivot_table('HOTELNIGHTS', 'COUNTRYOFRESIDENCE', 'MONTH')
timestamps_colnames = train_piv_hdf.drop('COUNTRYOFRESIDENCE').columns
timestamps_colnames.sort()
train_piv_hdf = train_piv_hdf.select('COUNTRYOFRESIDENCE', timestamps_colnames)
train_piv_hdf = train_piv_hdf.sort_values('COUNTRYOFRESIDENCE')
train_piv_hdf.head(5).collect()

,COUNTRYOFRESIDENCE,2022-01-01,2022-02-01,2022-03-01,2022-04-01,2022-05-01,2022-06-01,2022-07-01,2022-08-01,2022-09-01,...,2024-06-01,2024-07-01,2024-08-01,2024-09-01,2024-10-01,2024-11-01,2024-12-01,2025-01-01,2025-02-01,2025-03-01
0,Argentina,1410,1201,1880,2393,3315,3973,4563,4464,4522,...,6227,5301,5115,6191,4978,2433,2644,5333,4672,3944
1,Australia,5029,3797,6103,6656,10662,18501,24910,22174,25951,...,44598,48367,39527,52915,34824,14772,49368,44640,15890,16426
2,Austria,14588,16512,24802,22223,27612,34348,39430,46100,35715,...,34636,39039,43372,36691,30417,22880,22095,22414,26147,27781
3,Bahrain,294,337,474,298,903,1005,2570,3323,1077,...,1298,2188,2729,993,958,462,1924,730,1543,412
4,Belarus,288,358,383,431,958,878,1027,974,791,...,737,1009,896,841,669,458,632,585,595,609


Replace missing values (nan) with 0. This is needed for the payload formatting of the API call. "Out of range float values are not JSON compliant: nan"

In [16]:

train_piv_hdf = train_piv_hdf.fillna(0)

Keep list of the time-series names for later

In [17]:
ts_names_df = train_piv_hdf.select('COUNTRYOFRESIDENCE').collect()
ts_names_df.head(5)

,COUNTRYOFRESIDENCE
0,Argentina
1,Australia
2,Austria
3,Bahrain
4,Belarus


Download the data

In [18]:
train_df = train_piv_hdf.collect()
train_df.head(5)

,COUNTRYOFRESIDENCE,2022-01-01,2022-02-01,2022-03-01,2022-04-01,2022-05-01,2022-06-01,2022-07-01,2022-08-01,2022-09-01,...,2024-06-01,2024-07-01,2024-08-01,2024-09-01,2024-10-01,2024-11-01,2024-12-01,2025-01-01,2025-02-01,2025-03-01
0,Argentina,1410,1201,1880,2393,3315,3973,4563,4464,4522,...,6227,5301,5115,6191,4978,2433,2644,5333,4672,3944
1,Australia,5029,3797,6103,6656,10662,18501,24910,22174,25951,...,44598,48367,39527,52915,34824,14772,49368,44640,15890,16426
2,Austria,14588,16512,24802,22223,27612,34348,39430,46100,35715,...,34636,39039,43372,36691,30417,22880,22095,22414,26147,27781
3,Bahrain,294,337,474,298,903,1005,2570,3323,1077,...,1298,2188,2729,993,958,462,1924,730,1543,412
4,Belarus,288,358,383,431,958,878,1027,974,791,...,737,1009,896,841,669,458,632,585,595,609


Convert columns of type Decimal to Float. This is needed for the payload formatting of the API call. "Object of type Decimal is not JSON serializable"

In [19]:
from decimal import Decimal
for col in train_df.select_dtypes(include=['object']).columns:
    if all(isinstance(x, Decimal) for x in train_df[col].dropna()):
        #print(f"Column {col} is of type Decimal")
        train_df[col] = train_df[col].astype(float)        

/tmp/ipykernel_5412/2786318176.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in train_df.select_dtypes(include=['object']).columns:


Call SAP-RPT-TS to obtain the prediction

In [20]:
import requests
import os

# Payload data
payload = {
    "horizon": 12,
    "data": [{
        "timestamp": timestamps_colnames,
        "target":    train_df.drop('COUNTRYOFRESIDENCE', axis=1).values.tolist()
    }],
    "options": {"confidence_interval": 0.95},
}

# Headers
headers = {
    'AI-Resource-Group': 'default',
    'Authorization': f'Bearer {AUTH_TOKEN}',
    'Content-Type': 'application/json'
}

# Make the POST request
response = requests.post(
    f"{DEPLOYMENT_URL_SAPRPTTS}/predict",
    headers=headers,
    json=payload
)

# Print the response
print(response.status_code)

200


In [21]:
response.json()

{'all_null_targets': [],
 'confidence_interval_output': [[[[2877.057861328125,
     3921.533447265625,
     4436.42822265625,
     3975.891357421875,
     3877.658447265625,
     4151.95068359375,
     3133.4248046875,
     912.2796630859375,
     728.4293212890625,
     2826.711181640625,
     2225.91162109375,
     1972.1748046875],
    [6208.3154296875,
     7995.39990234375,
     9587.806640625,
     8728.5244140625,
     9159.654296875,
     9733.271484375,
     8368.298828125,
     5606.3193359375,
     5681.26806640625,
     8416.609375,
     7976.1279296875,
     7268.3369140625]],
   [[9866.1064453125,
     19624.337890625,
     29151.5234375,
     32904.734375,
     24498.884765625,
     30988.1328125,
     13284.16796875,
     2405.587158203125,
     27130.95703125,
     23835.2265625,
     2004.0850830078125,
     2197.0966796875],
    [37377.26171875,
     53043.0859375,
     64594.2109375,
     71097.3203125,
     64263.8125,
     74111.484375,
     59545.41796875,
     3

### Format and analyse the prediction data

Collect the prediction and the lower/upper bounds in a Pandas DataFrame

In [22]:
import pandas as pd
yhat = pd.DataFrame(response.json()['result'][0])
yhat.columns = response.json()['result_timestamp'][0]
yhat

,2025-04-01,2025-05-01,2025-06-01,2025-07-01,2025-08-01,2025-09-01,2025-10-01,2025-11-01,2025-12-01,2026-01-01,2026-02-01,2026-03-01
0,4495.290039,5791.452148,6843.326172,6034.686523,6074.958008,6631.292480,5390.831543,3406.758545,3359.466309,5253.605957,4841.570312,4454.226074
1,24189.779297,37964.906250,47247.320312,51647.171875,43824.535156,52366.554688,36896.324219,19953.476562,51009.179688,47119.843750,21176.337891,21572.199219
2,25553.675781,30925.470703,36089.675781,41790.984375,44605.558594,37100.324219,29284.673828,23453.943359,22674.347656,23273.080078,25775.947266,28753.111328
3,679.807434,885.158203,1265.807129,2525.722168,3083.958008,1229.404663,1065.682129,655.275757,1332.781006,774.196533,1157.042847,713.756592
4,627.508057,698.659607,715.819580,875.963623,884.152100,713.497009,620.290649,527.025024,611.920288,608.517090,605.313477,780.074768
...,...,...,...,...,...,...,...,...,...,...,...,...
67,17848.441406,18291.156250,20098.085938,18869.595703,17290.314453,17328.421875,17081.640625,20118.572266,26584.933594,26552.367188,20986.156250,20482.523438
68,7009.581055,7888.499512,8036.302734,9942.748047,9684.072266,7261.698242,7132.295410,6692.003906,10529.206055,11405.378906,8329.021484,7626.608398
69,15789.028320,19690.583984,24200.052734,50423.210938,55363.382812,19749.958984,20005.355469,14401.087891,24163.648438,17920.289062,18757.906250,17174.302734
70,108179.734375,119098.703125,152705.531250,169740.125000,177026.125000,150807.421875,101326.765625,81305.750000,151452.406250,135157.937500,175272.593750,149933.625000


Add column with name of time-series

In [23]:
yhat.insert(0, 'COUNTRYOFRESIDENCE', ts_names_df['COUNTRYOFRESIDENCE'])
yhat

,COUNTRYOFRESIDENCE,2025-04-01,2025-05-01,2025-06-01,2025-07-01,2025-08-01,2025-09-01,2025-10-01,2025-11-01,2025-12-01,2026-01-01,2026-02-01,2026-03-01
0,Argentina,4495.290039,5791.452148,6843.326172,6034.686523,6074.958008,6631.292480,5390.831543,3406.758545,3359.466309,5253.605957,4841.570312,4454.226074
1,Australia,24189.779297,37964.906250,47247.320312,51647.171875,43824.535156,52366.554688,36896.324219,19953.476562,51009.179688,47119.843750,21176.337891,21572.199219
2,Austria,25553.675781,30925.470703,36089.675781,41790.984375,44605.558594,37100.324219,29284.673828,23453.943359,22674.347656,23273.080078,25775.947266,28753.111328
3,Bahrain,679.807434,885.158203,1265.807129,2525.722168,3083.958008,1229.404663,1065.682129,655.275757,1332.781006,774.196533,1157.042847,713.756592
4,Belarus,627.508057,698.659607,715.819580,875.963623,884.152100,713.497009,620.290649,527.025024,611.920288,608.517090,605.313477,780.074768
...,...,...,...,...,...,...,...,...,...,...,...,...,...
67,Turkey,17848.441406,18291.156250,20098.085938,18869.595703,17290.314453,17328.421875,17081.640625,20118.572266,26584.933594,26552.367188,20986.156250,20482.523438
68,Ukraine,7009.581055,7888.499512,8036.302734,9942.748047,9684.072266,7261.698242,7132.295410,6692.003906,10529.206055,11405.378906,8329.021484,7626.608398
69,United Arab Emirates,15789.028320,19690.583984,24200.052734,50423.210938,55363.382812,19749.958984,20005.355469,14401.087891,24163.648438,17920.289062,18757.906250,17174.302734
70,United Kingdom,108179.734375,119098.703125,152705.531250,169740.125000,177026.125000,150807.421875,101326.765625,81305.750000,151452.406250,135157.937500,175272.593750,149933.625000


Confidence intervals are available, in this example they are not used any further

In [24]:
response.json()['confidence_interval_output'][0]

[[[2877.057861328125,
   3921.533447265625,
   4436.42822265625,
   3975.891357421875,
   3877.658447265625,
   4151.95068359375,
   3133.4248046875,
   912.2796630859375,
   728.4293212890625,
   2826.711181640625,
   2225.91162109375,
   1972.1748046875],
  [6208.3154296875,
   7995.39990234375,
   9587.806640625,
   8728.5244140625,
   9159.654296875,
   9733.271484375,
   8368.298828125,
   5606.3193359375,
   5681.26806640625,
   8416.609375,
   7976.1279296875,
   7268.3369140625]],
 [[9866.1064453125,
   19624.337890625,
   29151.5234375,
   32904.734375,
   24498.884765625,
   30988.1328125,
   13284.16796875,
   2405.587158203125,
   27130.95703125,
   23835.2265625,
   2004.0850830078125,
   2197.0966796875],
  [37377.26171875,
   53043.0859375,
   64594.2109375,
   71097.3203125,
   64263.8125,
   74111.484375,
   59545.41796875,
   38554.953125,
   78291.609375,
   73710.3125,
   43512.84765625,
   43961.2421875]],
 [[18195.470703125,
   23544.759765625,
   27737.21875,
   

### Calculate the MAPE for each time-series

For the accuracy calculation bring the time-series from row-level format to column

In [25]:
holdout_timestamps_df = holdout_hdf.select('MONTH').distinct().sort_values('MONTH').collect()
holdout_timestamps_df['MONTH'] = holdout_timestamps_df['MONTH'].astype(str)
holdout_timestamps_colnames = holdout_timestamps_df['MONTH'].values
holdout_timestamps_colnames

<ArrowStringArray>
['2025-04-01', '2025-05-01', '2025-06-01', '2025-07-01', '2025-08-01',
 '2025-09-01', '2025-10-01', '2025-11-01', '2025-12-01', '2026-01-01',
 '2026-02-01', '2026-03-01']
Length: 12, dtype: str

In [26]:
predictions_df = pd.melt(yhat, id_vars=["COUNTRYOFRESIDENCE"], value_vars=holdout_timestamps_colnames)
predictions_df.columns= ['COUNTRYOFRESIDENCE', 'MONTH', 'PREDICTION']
predictions_df

,COUNTRYOFRESIDENCE,MONTH,PREDICTION
0,Argentina,2025-04-01,4495.290039
1,Australia,2025-04-01,24189.779297
2,Austria,2025-04-01,25553.675781
3,Bahrain,2025-04-01,679.807434
4,Belarus,2025-04-01,627.508057
...,...,...,...
859,Turkey,2026-03-01,20482.523438
860,Ukraine,2026-03-01,7626.608398
861,United Arab Emirates,2026-03-01,17174.302734
862,United Kingdom,2026-03-01,149933.625000


In [27]:
holdout_hdf.count()

864

Retrieve holdout data

In [28]:
holdout_df = holdout_hdf.collect()
holdout_df['MONTH'] = holdout_df['MONTH'].astype(str)

Join actuals with predictions

In [29]:
holdout_and_predictions_df = pd.merge(
    left=holdout_df, 
    right=predictions_df,
    how='inner',
    left_on=['COUNTRYOFRESIDENCE', 'MONTH'],
    right_on=['COUNTRYOFRESIDENCE', 'MONTH']
)
holdout_and_predictions_df.head(5)

,COUNTRYOFRESIDENCE,MONTH,HOTELNIGHTS,PREDICTION
0,Argentina,2025-04-01,5665,4495.290039
1,Argentina,2025-05-01,8168,5791.452148
2,Argentina,2025-06-01,7989,6843.326172
3,Argentina,2025-07-01,8464,6034.686523
4,Argentina,2025-08-01,7385,6074.958008


Calculate the MAPE for each time-series

In [30]:
holdout_and_predictions_df['APE'] = abs((holdout_and_predictions_df['HOTELNIGHTS'] - holdout_and_predictions_df['PREDICTION']))  / holdout_and_predictions_df['HOTELNIGHTS']
mape_per_country_df = holdout_and_predictions_df.groupby('COUNTRYOFRESIDENCE')['APE'].mean().reset_index()
mape_per_country_df.columns = ['COUNTRYOFRESIDENCE', 'MAPE']
mape_per_country_df

,COUNTRYOFRESIDENCE,MAPE
0,Argentina,0.128870
1,Australia,0.137650
2,Austria,0.036740
3,Bahrain,0.375247
4,Belarus,0.374815
...,...,...
67,Turkey,0.186135
68,Ukraine,0.177204
69,United Arab Emirates,0.141471
70,United Kingdom,0.060802


Calculate the average MAPE

In [31]:
mape_per_country_df[['MAPE']].mean()

MAPE    0.159937
dtype: float64

Visualise the different MAPEs in a boxplot

In [32]:
fig = px.box(mape_per_country_df, x='MAPE', title='MAPE Distribution across Countries', points='all', hover_data=['COUNTRYOFRESIDENCE'])
fig.show()

### Save predictions to SAP HANA Cloud

In [33]:
data_hdf = dataframe.create_dataframe_from_pandas(
    connection_context=conn,
    pandas_df=holdout_and_predictions_df,
    table_name='HOTELNIGHTS_FORECAST_MONTHLY_BYCOUNTRY_SAPRPTTS',
    force=True,
    drop_exist_tab=True)

100%|██████████| 1/1 [00:00<00:00, 46.65it/s]
